# Deep Learning Steganography — Image-in-Image
### Final Project | Group 2 | Lê Thị Hồng Hạnh (22127103)

**Keywords:** Deep Learning Steganography, Image-in-Image, Encoder-Decoder, U-Net, Attention Mechanism (CBAM)

---

## Notebook Structure

| # | Section | Content |
|---|---------|---------|
| 1 | Setup | Install packages, imports |
| 2 | Configuration | All hyperparameters in one place |
| 3 | Dataset | ImageNet loading, preprocessing, pairing |
| 4 | Networks | PrepNetwork, CBAM, HidingNetwork (U-Net), RevealNetwork, StegaNet |
| 5 | Loss Functions | MSE Cover + MSE Secret + Perceptual (VGG-16) |
| 6 | Metrics | PSNR, SSIM, LPIPS |
| 7 | Training | Full train/val loop with checkpointing |
| 8 | Inference & Demo | Hide + Reveal + visualize results |


## Section 1 — Setup

In [2]:
# Install required packages (run once)
!pip install torch torchvision Pillow numpy matplotlib tqdm

In [3]:
import os
import sys
import random
import time
import math
from pathlib import Path
from typing import Optional, Tuple, Callable

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as T
import torchvision.utils as vutils

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

# ── Device ──────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  Running on CPU")

Using device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Section 2 — Configuration

All hyperparameters in one place. Adjust here before training.


In [7]:
from dataclasses import dataclass, field

@dataclass
class ModelConfig:
    """Network architecture hyperparameters."""
    prep_out_ch:  int = 16    # Output channels of PrepNetwork
    unet_base_ch: int = 32    # Base channel width of U-Net
    image_size:   int = 256   # Input/output spatial resolution


@dataclass
class LossConfig:
    """Loss weights:
       L = alpha*MSE_cover + beta*MSE_secret + beta_ssim*SSIM_secret
         + gamma*Percep_cover + delta*Percep_secret
    """
    alpha:     float = 1.0    # Cover MSE weight
    beta:      float = 1.0    # Secret MSE weight
    beta_ssim: float = 0.5    # Secret SSIM loss weight
    gamma:     float = 0.1    # Cover perceptual loss weight (VGG-16)
    delta:     float = 0.05   # Secret perceptual loss weight (VGG-16)


@dataclass
class TrainConfig:
    """Training loop and optimizer settings."""
    # ── Dataset paths ———
    train_root:   str = "/content/drive/MyDrive/image-in-image/data/imagenet/train"
    val_root:     str = "/content/drive/MyDrive/image-in-image/data/imagenet/val"
    train_max_size: int = 5000   # Cap training images
    val_max_size:   int = 200    # Cap val images

    # ── Training ──
    epochs:       int   = 10
    batch_size:   int   = 8
    num_workers:  int   = 4

    # ── Optimizer ──
    lr:           float = 1e-4
    weight_decay: float = 1e-5
    lr_step_size: int   = 10      # Reduce LR every N epochs
    lr_gamma:     float = 0.5

    # ── Checkpointing ──
    checkpoint_dir: str = "/content/drive/MyDrive/image-in-image2/checkpoints"
    save_every:     int = 3       # Save every 3 epochs
    resume_from:    Optional[str] = None

    # ── Logging & Visualization ──
    log_every:     int = 50       # Print loss every N batches
    val_every:     int = 1
    vis_every:     int = 3
    vis_dir:       str = "/content/drive/MyDrive/image-in-image2/outputs/vis"
    n_vis_samples: int = 4


@dataclass
class Config:
    model: ModelConfig = field(default_factory=ModelConfig)
    loss:  LossConfig  = field(default_factory=LossConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

cfg = Config()

print(f"Prep channels    : {cfg.model.prep_out_ch}")
print(f"U-Net base ch    : {cfg.model.unet_base_ch}")
print(f"Epochs           : {cfg.train.epochs}")
print(f"Batch size       : {cfg.train.batch_size}")
print(f"Train max images : {cfg.train.train_max_size}")
print(f"Val max images   : {cfg.train.val_max_size}")
print(f"Loss α/β/β_ssim/γ/δ : {cfg.loss.alpha}/{cfg.loss.beta}/{cfg.loss.beta_ssim}/{cfg.loss.gamma}/{cfg.loss.delta}")

Prep channels    : 16
U-Net base ch    : 32
Epochs           : 10
Batch size       : 8
Train max images : 5000
Val max images   : 200
Loss α/β/β_ssim/γ/δ : 1.0/1.0/0.5/0.1/0.05


---
## Section 3 — Dataset

**Preprocessing pipeline**:
1. Resize all images to 256×256
2. Normalize pixel values to [0, 1] via `ToTensor()`
3. Randomly pair cover and secret images (always different images)

**Supported datasets:** ImageNet 256×256


In [8]:
SUPPORTED_EXTS = {".jpg"}

def find_images(root: str) -> list:
    """Recursively find all image files under root directory."""
    root = Path(root)
    images = [str(p) for p in root.rglob("*") if p.suffix.lower() in SUPPORTED_EXTS]
    images.sort()
    return images


def get_transform(split: str = "train", size: int = 256) -> T.Compose:
    """
    Train: Resize + RandomHorizontalFlip + ToTensor [0,1]
    Val  : Resize + ToTensor [0,1]
    Note: NO ImageNet mean/std normalization — model works with raw [0,1] values.
    """
    if split == "train":
        return T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(p=0.5), T.ToTensor()])
    else:
        return T.Compose([T.Resize((size, size)), T.ToTensor()])


class StegoDataset(Dataset):
    """
    Generic steganography dataset.
    Each __getitem__ returns (cover, secret) — always two DIFFERENT images.
    Works with ImageNet, COCO, or any folder of images.
    """

    def __init__(self, root: str, split: str = "train", size: int = 256,
                 transform=None, max_size: Optional[int] = None):
        self.images = find_images(root)
        if not self.images:
            raise FileNotFoundError(f"No images found under '{root}'.")
        if max_size is not None:
            self.images = self.images[:max_size]
        self.transform = transform or get_transform(split, size)
        self.n = len(self.images)
        print(f"[StegoDataset] {split}: {self.n} images from '{root}'")

    def __len__(self):
        return self.n

    def _load(self, path: str) -> torch.Tensor:
        return self.transform(Image.open(path).convert("RGB"))

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        cover = self._load(self.images[idx])
        # Pick a DIFFERENT image as the secret
        secret_idx = idx
        while secret_idx == idx:
            secret_idx = random.randint(0, self.n - 1)
        secret = self._load(self.images[secret_idx])
        return cover, secret


def build_dataloaders(cfg: Config) -> Tuple[DataLoader, DataLoader]:
    """Build train and validation DataLoaders from config."""
    pin_memory = torch.cuda.is_available()

    train_ds = StegoDataset(cfg.train.train_root, split="train",
                          size=cfg.model.image_size, max_size=cfg.train.train_max_size)
    val_ds   = StegoDataset(cfg.train.val_root,   split="val",
                          size=cfg.model.image_size, max_size=cfg.train.val_max_size)

    train_loader = DataLoader(train_ds, batch_size=cfg.train.batch_size,
                              shuffle=True, num_workers=cfg.train.num_workers,
                              pin_memory=pin_memory, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.train.batch_size,
                              shuffle=False, num_workers=cfg.train.num_workers,
                              pin_memory=pin_memory)
    return train_loader, val_loader

In [9]:
# ── Quick dataset verification ───────────────────────────────

train_loader, val_loader = build_dataloaders(cfg)
cover, secret = next(iter(train_loader))
print(f"Cover  shape : {cover.shape}")    # (B, 3, 256, 256)
print(f"Secret shape : {secret.shape}")   # (B, 3, 256, 256)
print(f"Value range  : [{cover.min():.2f}, {cover.max():.2f}]")   # [0.0, 1.0]
print(f"Same image?  : {(cover == secret).all()}")                 # False

print("Dataset classes defined. Build dataloaders after setting your data paths.")

[StegoDataset] train: 5000 images from '/content/drive/MyDrive/image-in-image/data/imagenet/train'
[StegoDataset] val: 200 images from '/content/drive/MyDrive/image-in-image/data/imagenet/val'


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Cover  shape : torch.Size([8, 3, 256, 256])
Secret shape : torch.Size([8, 3, 256, 256])
Value range  : [0.00, 1.00]
Same image?  : False
Dataset classes defined. Build dataloaders after setting your data paths.


---
## Section 4 — Network Architectures

### Architecture Overview
```
Secret S  ---> PrepNetwork --> PrepOut (16 ch)
                                    │
Cover C   -------------------> Concatenate (16+3 ch)
                                    │
                                    V
                          HidingNetwork (U-Net + CBAM)
                                    │
                            Stego C' -----------------> [Output]
                                    │
                                    V
                          RevealNetwork (U-Net)
                                    │
                             Revealed S' -------------> [Output]
```

### Sub-components
- **DoubleConv** — standard U-Net building block (Conv→BN→LeakyReLU ×2)
- **ChannelAttention** — CBAM: "WHAT features matter?" (channel-wise MLP)
- **SpatialAttention** — CBAM: "WHERE to embed?" (spatial conv)
- **CBAM** — Channel + Spatial attention combined
- **PrepNetwork** — Lightweight CNN: secret (3ch) → rich features (16 ch)
- **HidingNetwork** — U-Net + CBAM: (cover + prep) → stego image
- **RevealNetwork** — U-Net + CBAM: stego → revealed secret
- **StegaNet** — Full system combining all three


In [10]:
# ── DoubleConv: standard U-Net building block ────────────────
class DoubleConv(nn.Module):
    """Two consecutive Conv2d → BatchNorm → LeakyReLU blocks."""

    def __init__(self, in_ch: int, out_ch: int, mid_ch: int = None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(mid_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.block(x)

print("DoubleConv defined.")

DoubleConv defined.


In [11]:
# ── CBAM: Convolutional Block Attention Module ───────────────

class ChannelAttention(nn.Module):
    """
    Answers: WHAT features are important?
    Pools spatially (avg + max), passes through shared MLP, applies sigmoid.
    """
    def __init__(self, in_ch: int, reduction: int = 16):
        super().__init__()
        reduced = max(1, in_ch // reduction)
        self.shared_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, in_ch, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        avg_pool = x.mean(dim=(2, 3))       # (B, C)
        max_pool = x.amax(dim=(2, 3))       # (B, C)
        attention = self.sigmoid(self.shared_mlp(avg_pool) + self.shared_mlp(max_pool))
        return x * attention.view(B, C, 1, 1)


class SpatialAttention(nn.Module):
    """
    Answers: WHERE is important information located?
    Aggregates channels (avg + max), applies 7×7 conv, sigmoid.
    """
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size,
                              padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = x.mean(dim=1, keepdim=True)   # (B, 1, H, W)
        max_out = x.amax(dim=1, keepdim=True)   # (B, 1, H, W)
        scale = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * scale


class CBAM(nn.Module):
    """
    Full CBAM: ChannelAttention → SpatialAttention applied sequentially.
    Drop-in module for any feature map tensor.
    """
    def __init__(self, in_ch: int, reduction: int = 16, spatial_kernel: int = 7):
        super().__init__()
        self.channel_att = ChannelAttention(in_ch, reduction=reduction)
        self.spatial_att = SpatialAttention(kernel_size=spatial_kernel)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

print("CBAM (ChannelAttention + SpatialAttention) defined.")

CBAM (ChannelAttention + SpatialAttention) defined.


In [12]:
# ── PrepNetwork ──────────────────────────────────────────────
# Transforms secret image 3ch -> N ch for richer feature representation

class PrepNetwork(nn.Module):
    """
    Lightweight CNN: expands secret image from 3ch to N channels.
    All convolutions preserve spatial dims (256×256) via padding=1.

    Input : (B, 3, H, W)    — secret image
    Output: (B, N, H, W)    — richer feature representation
    """
    def __init__(self, in_ch: int = 3, out_ch: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, secret: torch.Tensor) -> torch.Tensor:
        return self.net(secret)

print("PrepNetwork defined.")

PrepNetwork defined.


In [14]:
# ── HidingNetwork (U-Net + CBAM) ─────────────────────────────
# CBAM placement rationale:
#   - Bottleneck (16×16): global embedding decisions across full image context
#   - dec3 (64×64): mid-level texture decisions — best scale for hiding

class HidingNetwork(nn.Module):
    """
    U-Net encoder-decoder with CBAM attention.

    Input : concat([Cover C (3ch), PrepOut (16 ch)]) → (16+3, 256, 256)
    Output: Stego image C' (3, 256, 256)

    Encoder:  (16+3)→64 → 128 → 256 → 512 → bottleneck 1024 (with CBAM)
    Decoder:  1024 → 512 → 256 (with CBAM) → 128 → 64 → 3
    """
    def __init__(self, in_ch: int = 19, base_ch: int = 32):
        super().__init__()
        b = base_ch

        # Encoder
        self.enc1 = DoubleConv(in_ch, b)       # 256×256
        self.enc2 = DoubleConv(b, b * 2)       # 128×128
        self.enc3 = DoubleConv(b * 2, b * 4)   # 64×64
        self.enc4 = DoubleConv(b * 4, b * 8)   # 32×32
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck + CBAM
        self.bottleneck      = DoubleConv(b * 8, b * 16)   # 16×16
        self.cbam_bottleneck = CBAM(b * 16)

        # Decoder
        self.up4   = nn.ConvTranspose2d(b * 16, b * 8, kernel_size=2, stride=2)
        self.dec4  = DoubleConv(b * 16, b * 8)

        self.up3      = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec3     = DoubleConv(b * 8, b * 4)
        self.cbam_dec3 = CBAM(b * 4)             # CBAM at 64×64 level

        self.up2   = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec2  = DoubleConv(b * 4, b * 2)

        self.up1   = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec1  = DoubleConv(b * 2, b)

        # Output
        self.out_conv = nn.Conv2d(b, 3, kernel_size=1)
        self.out_act  = nn.Sigmoid()

    def forward(self, cover: torch.Tensor, prep_features: torch.Tensor) -> torch.Tensor:
        x = torch.cat([cover, prep_features], dim=1)   # (B, 16+3, H, W)

        # Encode
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        bn = self.cbam_bottleneck(self.bottleneck(self.pool(e4)))

        # Decode with skip connections
        d4 = self.dec4(torch.cat([self.up4(bn), e4], dim=1))
        d3 = self.cbam_dec3(self.dec3(torch.cat([self.up3(d4), e3], dim=1)))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_act(self.out_conv(d1))

print("HidingNetwork (U-Net + CBAM) defined.")

HidingNetwork (U-Net + CBAM) defined.


In [15]:
# ── RevealNetwork (U-Net + CBAM) ─────────────────────────────
# Add CBAM to RevealNetwork so it learns to focus on the
# same textured regions where HidingNetwork embedded the secret signal.
# CBAM placement mirrors HidingNetwork: bottleneck + dec3 level.

class RevealNetwork(nn.Module):
    """
    U-Net encoder-decoder with CBAM attention for secret extraction.
    CBAM helps the network focus on textured regions where the hidden
    signal was embedded, mirroring the HidingNetwork\'s attention pattern.

    Input : Stego image C\' (3, 256, 256)
    Output: Revealed secret S\' (3, 256, 256)
    """
    def __init__(self, base_ch: int = 32):
        super().__init__()
        b = base_ch

        self.enc1 = DoubleConv(3, b)
        self.enc2 = DoubleConv(b, b * 2)
        self.enc3 = DoubleConv(b * 2, b * 4)
        self.enc4 = DoubleConv(b * 4, b * 8)
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck + CBAM: mirrors HidingNetwork\'s bottleneck attention
        self.bottleneck      = DoubleConv(b * 8, b * 16)
        self.cbam_bottleneck = CBAM(b * 16)

        self.up4  = nn.ConvTranspose2d(b * 16, b * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(b * 16, b * 8)

        self.up3       = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec3      = DoubleConv(b * 8, b * 4)
        self.cbam_dec3 = CBAM(b * 4)   # CBAM at 64x64: mirrors HidingNetwork\'s dec3

        self.up2  = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(b * 4, b * 2)
        self.up1  = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(b * 2, b)

        self.out_conv = nn.Conv2d(b, 3, kernel_size=1)
        self.out_act  = nn.Sigmoid()

    def forward(self, stego: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(stego)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck with CBAM
        bn = self.cbam_bottleneck(self.bottleneck(self.pool(e4)))

        d4 = self.dec4(torch.cat([self.up4(bn), e4], dim=1))
        d3 = self.cbam_dec3(self.dec3(torch.cat([self.up3(d4), e3], dim=1)))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_act(self.out_conv(d1))

print("RevealNetwork (U-Net + CBAM) defined.")


RevealNetwork (U-Net + CBAM) defined.


In [16]:
# ── StegaNet: Full System ─────────────────────────────────────

class StegaNet(nn.Module):
    """
    Complete image-in-image steganography system.

    Encoder path: secret → PrepNetwork → prep_feat
                  (cover, prep_feat) → HidingNetwork → stego C'
    Decoder path: stego C' → RevealNetwork → revealed S'

    Usage:
        model = StegaNet(prep_out_ch=16, unet_base_ch=32)  # baseline
        stego, revealed = model(cover, secret)
    """
    def __init__(self, prep_out_ch: int = 16, unet_base_ch: int = 32):
        super().__init__()
        self.prep   = PrepNetwork(in_ch=3, out_ch=prep_out_ch)
        self.hiding = HidingNetwork(in_ch=3 + prep_out_ch, base_ch=unet_base_ch)
        self.reveal = RevealNetwork(base_ch=unet_base_ch)

    def encode(self, cover: torch.Tensor, secret: torch.Tensor) -> torch.Tensor:
        """Encoder only — produces stego image."""
        return self.hiding(cover, self.prep(secret))

    def decode(self, stego: torch.Tensor) -> torch.Tensor:
        """Decoder only — reveals hidden secret from stego."""
        return self.reveal(stego)

    def forward(self, cover: torch.Tensor, secret: torch.Tensor):
        """Full pipeline: (cover, secret) → (stego, revealed)"""
        prep_feat = self.prep(secret)
        stego     = self.hiding(cover, prep_feat)
        revealed  = self.reveal(stego)
        return stego, revealed


# ── Sanity check ─────────────────────────────────────────────
model = StegaNet(
    prep_out_ch=cfg.model.prep_out_ch,
    unet_base_ch=cfg.model.unet_base_ch
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"StegaNet created successfully!")
print(f"  prep_out_ch  : {cfg.model.prep_out_ch}")
print(f"  unet_base_ch : {cfg.model.unet_base_ch}")
print(f"  Total params : {n_params:,}")

# Forward pass test
with torch.no_grad():
    _cover  = torch.randn(1, 3, 256, 256).to(device)
    _secret = torch.randn(1, 3, 256, 256).to(device)
    _stego, _revealed = model(_cover, _secret)
    print(f"  Stego shape    : {_stego.shape}")
    print(f"  Revealed shape : {_revealed.shape}")
    print(f"  Stego range    : [{_stego.min():.3f}, {_stego.max():.3f}]  ← should be [0,1]")
del _cover, _secret, _stego, _revealed

StegaNet created successfully!
  prep_out_ch  : 16
  unet_base_ch : 32
  Total params : 15,648,942
  Stego shape    : torch.Size([1, 3, 256, 256])
  Revealed shape : torch.Size([1, 3, 256, 256])
  Stego range    : [0.113, 0.903]  ← should be [0,1]


---
## Section 5 — Loss Functions

$$\mathcal{L}_{Total} = \alpha \cdot \mathcal{L}_{MSE\_Cover} + \beta \cdot \mathcal{L}_{MSE\_Secret} (+ \beta_{ssim} \cdot \mathcal{L}_{SSIM\_Secret}) + \gamma \cdot \mathcal{L}_{Percep\_Cover} + \delta \cdot \mathcal{L}_{Percep\_Secret}$$

| Term | Formula | Purpose |
|:-----|:--------|:--------|
| $\mathcal{L}_{MSE\_Cover}$ | $||C - C'||^2_2$ | Pixel-level invisibility |
| $\mathcal{L}_{MSE\_Secret}$ | $||S - S'||^2_2$ | Pixel-level recovery accuracy |
| $\mathcal{L}_{SSIM\_Secret}$ | $1-SSIM(S, S')$ | Structural recovery quality |
| $\mathcal{L}_{Percep\_Cover}$ | $|| \phi(C)-\phi(C')||^2_2$ | VGG-16 perceptual cover quality |
| $\mathcal{L}_{Percep\_Secret}$ | $||\phi(S)-\phi(S')||^2_2$ | VGG-16 perceptual secret recovery |
              
                      
  


In [17]:
# ── Loss Functions ─────────────────────────────

class PerceptualLoss(nn.Module):
    """
    VGG-16 perceptual loss — reusable for both cover and secret pairs.
    Compares high-level feature representations (relu1_2, relu2_2, relu3_3).
    Ref: Zeng et al. (2023)
    """
    VGG_LAYERS = {"relu1_1": 1,"relu1_2": 4, "relu2_2": 9, "relu3_3": 16}

    def __init__(self, layer_weights: dict = None):
        super().__init__()
        if layer_weights is None:
            layer_weights = {k: 1.0 / len(self.VGG_LAYERS) for k in self.VGG_LAYERS}
        self.layer_weights = layer_weights

        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        vgg.eval()
        for param in vgg.parameters():
            param.requires_grad = False

        self.extractors = nn.ModuleDict()
        for name, stop_idx in self.VGG_LAYERS.items():
            self.extractors[name] = nn.Sequential(*list(vgg.features.children())[:stop_idx + 1])

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, img_a: torch.Tensor, img_b: torch.Tensor) -> torch.Tensor:
        """Compute perceptual loss between any two image pairs (img_a, img_b)."""
        a_n, b_n = self._normalize(img_a), self._normalize(img_b)
        loss = torch.tensor(0.0, device=img_a.device)
        for name, extractor in self.extractors.items():
            w = self.layer_weights.get(name, 1.0)
            loss = loss + w * F.mse_loss(extractor(b_n), extractor(a_n))
        return loss


def ssim_loss(img1: torch.Tensor, img2: torch.Tensor,
              window_size: int = 11, C1: float = 1e-4, C2: float = 9e-4) -> torch.Tensor:
    """
    SSIM-based structural loss: L_ssim = 1 - SSIM(img1, img2).
    Range [0, 2], practically ~[0, 1]. Lower = better structural similarity.
    Penalizes blurring, contrast loss, and structural distortion that MSE misses.
    This directly optimises the SSIM metric used in evaluation.
    """
    B, C, H, W = img1.shape
    coords = torch.arange(window_size, dtype=torch.float32, device=img1.device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    g = g / g.sum()
    kernel = (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0).expand(C, 1, -1, -1)
    pad = window_size // 2

    def conv(x): return F.conv2d(x, kernel, padding=pad, groups=C)

    mu1, mu2   = conv(img1), conv(img2)
    sigma1_sq  = conv(img1 * img1) - mu1 ** 2
    sigma2_sq  = conv(img2 * img2) - mu2 ** 2
    sigma12    = conv(img1 * img2) - mu1 * mu2
    ssim_map   = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / \
                 ((mu1 ** 2 + mu2 ** 2 + C1) * (sigma1_sq + sigma2_sq + C2))
    return 1.0 - ssim_map.mean()


class SteganographyLoss(nn.Module):
    """
    Improved combined loss:
      L = alpha * MSE_cover + beta * MSE_secret
        + beta_ssim * SSIM_secret
        + gamma * Percep_cover + delta * Percep_secret

    Args:
        alpha     : weight for cover MSE (imperceptibility)
        beta_mse  : weight for secret pixel MSE (recovery accuracy)
        beta_ssim : weight for secret SSIM loss (structural quality)
        gamma     : weight for cover perceptual loss (VGG cover)
        delta     : weight for secret perceptual loss (VGG secret)
    """
    def __init__(self,
                 alpha:     float = 1.0,
                 beta_mse:  float = 1.0,
                 beta_ssim: float = 0.5,
                 gamma:     float = 0.1,
                 delta:     float = 0.05):
        super().__init__()
        self.alpha, self.beta_mse, self.beta_ssim = alpha, beta_mse, beta_ssim
        self.gamma, self.delta = gamma, delta
        self.perceptual = PerceptualLoss()
        self.mse = nn.MSELoss()

    def forward(self, cover, stego, secret, revealed) -> dict:
        cover_mse     = self.mse(stego, cover)
        secret_mse    = self.mse(revealed, secret)
        secret_ssim   = ssim_loss(secret, revealed)
        percep_cover  = self.perceptual(cover, stego)
        percep_secret = self.perceptual(secret, revealed)

        total = (self.alpha     * cover_mse
               + self.beta_mse  * secret_mse
               + self.beta_ssim * secret_ssim
               + self.gamma     * percep_cover
               + self.delta     * percep_secret)

        return {
            "total":         total,
            "cover_mse":     cover_mse,
            "secret_mse":    secret_mse,
            "secret_ssim":   secret_ssim,
            "percep_cover":  percep_cover,
            "percep_secret": percep_secret,
        }


criterion = SteganographyLoss(
    alpha=cfg.loss.alpha,
    beta_mse=cfg.loss.beta,
    beta_ssim=cfg.loss.beta_ssim,
    gamma=cfg.loss.gamma,
    delta=cfg.loss.delta,
).to(device)

print(f"Improved SteganographyLoss created:")
print(f"  alpha     = {cfg.loss.alpha}  (cover MSE)")
print(f"  beta_mse  = {cfg.loss.beta}  (secret MSE)")
print(f"  beta_ssim = {cfg.loss.beta_ssim}   (secret SSIM)")
print(f"  gamma     = {cfg.loss.gamma}  (cover perceptual)")
print(f"  delta     = {cfg.loss.delta}  (secret perceptual)")


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 133MB/s]


Improved SteganographyLoss created:
  alpha     = 1.0  (cover MSE)
  beta_mse  = 1.0  (secret MSE)
  beta_ssim = 0.5   (secret SSIM)
  gamma     = 0.1  (cover perceptual)
  delta     = 0.05  (secret perceptual)


---
## Section 6 — Evaluation Metrics

| Metric | Applied to | Target |
|--------|-----------|--------|
| **PSNR** | C vs C', S vs S' | >30 dB (cover), >25 dB (secret) |
| **SSIM** | C vs C', S vs S' | >0.95 (cover), >0.90 (secret) |
| **LPIPS** | C vs C', S vs S' | Low (cover), Low (secret) |


In [18]:
def psnr(img1: torch.Tensor, img2: torch.Tensor, max_val: float = 1.0) -> torch.Tensor:
    """Peak Signal-to-Noise Ratio. Higher = better. Formula: 10*log10(max²/MSE)"""
    mse = F.mse_loss(img1, img2, reduction="none").mean(dim=(1, 2, 3)).clamp(min=1e-10)
    return (10 * torch.log10(max_val ** 2 / mse)).mean()


def _gaussian_kernel(window_size: int = 11, sigma: float = 1.5) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g.unsqueeze(0) * g.unsqueeze(1)


def ssim(img1: torch.Tensor, img2: torch.Tensor,
         window_size: int = 11, C1: float = 1e-4, C2: float = 9e-4) -> torch.Tensor:
    """Structural Similarity Index. Range [0,1]. Higher = better."""
    B, C, H, W = img1.shape
    kernel = _gaussian_kernel(window_size).to(img1.device)
    kernel = kernel.unsqueeze(0).unsqueeze(0).expand(C, 1, -1, -1)
    pad = window_size // 2

    def conv(x): return F.conv2d(x, kernel, padding=pad, groups=C)

    mu1, mu2   = conv(img1), conv(img2)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1*mu2
    sigma1_sq  = conv(img1*img1) - mu1_sq
    sigma2_sq  = conv(img2*img2) - mu2_sq
    sigma12    = conv(img1*img2) - mu1_mu2
    ssim_map   = ((2*mu1_mu2+C1)*(2*sigma12+C2)) / ((mu1_sq+mu2_sq+C1)*(sigma1_sq+sigma2_sq+C2))
    return ssim_map.mean()


class LPIPS(nn.Module):
    """Learned Perceptual Image Patch Similarity using VGG-16. Lower = better."""
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        vgg.eval()
        for p in vgg.parameters(): p.requires_grad = False
        feats = vgg.features
        self.s1 = nn.Sequential(*list(feats.children())[:4])
        self.s2 = nn.Sequential(*list(feats.children())[4:9])
        self.s3 = nn.Sequential(*list(feats.children())[9:16])
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def _norm(self, x): return (x - self.mean) / self.std

    def forward(self, img1, img2):
        def nm(a, b):
            return F.mse_loss(F.normalize(a,dim=1), F.normalize(b,dim=1))
        x1, x2 = self._norm(img1), self._norm(img2)
        h1 = self.s1(x1); h2 = self.s1(x2)
        h3 = self.s2(h1); h4 = self.s2(h2)
        h5 = self.s3(h3); h6 = self.s3(h4)
        return nm(h1,h2) + nm(h3,h4) + nm(h5,h6)


class MetricsCalculator:
    """Computes PSNR, SSIM, LPIPS for both cover↔stego and secret↔revealed pairs."""
    def __init__(self, device="cpu"):
        self.device = device
        self.lpips_fn = LPIPS().to(device).eval()

    @torch.no_grad()
    def compute(self, cover, stego, secret, revealed) -> dict:
        return {
            "psnr_cover":   psnr(cover, stego).item(),
            "ssim_cover":   ssim(cover, stego).item(),
            "lpips_cover":  self.lpips_fn(cover.to(self.device), stego.to(self.device)).item(),
            "psnr_secret":  psnr(secret, revealed).item(),
            "ssim_secret":  ssim(secret, revealed).item(),
            "lpips_secret": self.lpips_fn(secret.to(self.device), revealed.to(self.device)).item(),
        }

    def format(self, m: dict) -> str:
        return (f"[Cover↔Stego]     PSNR={m['psnr_cover']:.2f}dB  "
                f"SSIM={m['ssim_cover']:.4f}  LPIPS={m['lpips_cover']:.4f}"
                f"[Secret↔Revealed] PSNR={m['psnr_secret']:.2f}dB  "
                f"SSIM={m['ssim_secret']:.4f}  LPIPS={m['lpips_secret']:.4f}")


metrics_calc = MetricsCalculator(device=str(device))
print("Metrics (PSNR, SSIM, LPIPS) defined.")

Metrics (PSNR, SSIM, LPIPS) defined.


---
## Section 7 — Training
The training loop:
1. For each batch: forward pass → compute loss → backprop → optimizer step
2. Every epoch: validate on val set → compute PSNR/SSIM/LPIPS
3. Save best checkpoint (by PSNR secret) and periodic checkpoints
4. Every few epochs: save a visualization grid (cover | stego | secret | revealed)


In [19]:
# ── Checkpoint helpers ────────────────────────────────────────

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({
        "epoch": epoch, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "metrics": metrics,
    }, path)
    print(f"  ✓ Checkpoint saved → {path}")


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state["model"])
    if optimizer and "optimizer" in state:
        optimizer.load_state_dict(state["optimizer"])
    if scheduler and "scheduler" in state:
        scheduler.load_state_dict(state["scheduler"])
    return state.get("epoch", 0), state.get("metrics", {})

In [20]:
# ── Visualization helper ──────────────────────────────────────

def show_comparison(cover, stego, secret, revealed, n=4, title=""):
    """Display a grid: Cover | Stego | Secret | Revealed in the notebook."""
    n = min(n, cover.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n))
    if n == 1: axes = axes[None]  # ensure 2D indexing
    col_titles = ["Cover (C)", "Stego (C')", "Secret (S)", "Revealed (S')"]
    for j, ct in enumerate(col_titles):
        axes[0, j].set_title(ct, fontsize=12, fontweight="bold")
    for i in range(n):
        for j, img in enumerate([cover, stego, secret, revealed]):
            axes[i, j].imshow(img[i].cpu().permute(1, 2, 0).clamp(0, 1).numpy())
            axes[i, j].axis("off")
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def save_vis_grid(cover, stego, secret, revealed, path, n=4):
    """Save grid image to disk."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    imgs = torch.cat([cover[:n], stego[:n], secret[:n], revealed[:n]], dim=0)
    grid = vutils.make_grid(imgs, nrow=n, normalize=False, padding=2)
    vutils.save_image(grid, path)

In [21]:
# ── Training functions ────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, epoch, cfg):
    model.train()
    totals = {"total": 0., "cover_mse": 0., "secret_mse": 0., "secret_ssim": 0., "percep_cover": 0., "percep_secret": 0.}
    n = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
    for batch_idx, (cover, secret) in enumerate(pbar):
        cover, secret = cover.to(device), secret.to(device)
        optimizer.zero_grad()
        stego, revealed = model(cover, secret)
        losses = criterion(cover, stego, secret, revealed)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k in totals: totals[k] += losses[k].item()
        n += 1
        if (batch_idx + 1) % cfg.train.log_every == 0:
            avg = totals["total"] / n
            pbar.set_postfix({"loss": f"{avg:.4f}"})
    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def validate(model, loader, criterion, metrics_calc, cfg):
    model.eval()
    totals  = {"total": 0., "cover_mse": 0., "secret_mse": 0., "secret_ssim": 0., "percep_cover": 0., "percep_secret": 0.}
    met_sum = {"psnr_cover": 0., "ssim_cover": 0., "lpips_cover": 0.,
               "psnr_secret": 0., "ssim_secret": 0., "lpips_secret": 0.}
    n = 0
    vis_batch = None
    for cover, secret in tqdm(loader, desc="Validating", leave=False):
        cover, secret = cover.to(device), secret.to(device)
        stego, revealed = model(cover, secret)
        losses  = criterion(cover, stego, secret, revealed)
        metrics = metrics_calc.compute(cover, stego, secret, revealed)
        for k in totals:  totals[k]  += losses[k].item()
        for k in met_sum: met_sum[k] += metrics[k]
        n += 1
        if vis_batch is None:
            vis_batch = (cover.cpu(), stego.cpu(), secret.cpu(), revealed.cpu())
    return ({k: v/n for k,v in totals.items()},
            {k: v/n for k,v in met_sum.items()},
            vis_batch)

In [ ]:
# ── Main Training Loop ────────────────────────────────────────

def run_training(cfg: Config):
    print("=" * 60)
    print(f"  Deep Steganography Training")
    print(f"  Device      : {device}")
    print(f"  Epochs      : {cfg.train.epochs}")
    print(f"  Batch size  : {cfg.train.batch_size}")
    print(f"  Train imgs  : {cfg.train.train_max_size}")
    print(f"  Val imgs    : {cfg.train.val_max_size}")
    print("=" * 60)

    # Build model
    model = StegaNet(
        prep_out_ch=cfg.model.prep_out_ch,
        unet_base_ch=cfg.model.unet_base_ch
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}")

    # Loss, optimizer, scheduler
    criterion = SteganographyLoss(
        alpha=cfg.loss.alpha,
        beta_mse=cfg.loss.beta,
        beta_ssim=cfg.loss.beta_ssim,
        gamma=cfg.loss.gamma,
        delta=cfg.loss.delta,
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train.lr,
                           weight_decay=cfg.train.weight_decay)
    scheduler = StepLR(optimizer, step_size=cfg.train.lr_step_size, gamma=cfg.train.lr_gamma)
    metrics_calc = MetricsCalculator(device=str(device))

    # Data
    train_loader, val_loader = build_dataloaders(cfg)

    # Resume
    start_epoch = 0
    if cfg.train.resume_from and os.path.exists(cfg.train.resume_from):
        start_epoch, _ = load_checkpoint(cfg.train.resume_from, model, optimizer, scheduler)
        print(f"Resumed from epoch {start_epoch}")
        start_epoch += 1

    # History for plotting
    history = {"train_loss": [], "val_loss": [],
               "psnr_cover": [], "psnr_secret": [],
               "ssim_cover": [], "ssim_secret": []}
    best_psnr_secret = 0.0

    for epoch in range(start_epoch, cfg.train.epochs):
        t0 = time.time()

        # Train
        train_losses = train_one_epoch(model, train_loader, criterion, optimizer, epoch, cfg)
        scheduler.step()

        # Validate
        val_losses, val_metrics, vis_batch = validate(
            model, val_loader, criterion, metrics_calc, cfg)

        # Record history
        history["train_loss"].append(train_losses["total"])
        history["val_loss"].append(val_losses["total"])
        history["psnr_cover"].append(val_metrics["psnr_cover"])
        history["psnr_secret"].append(val_metrics["psnr_secret"])
        history["ssim_cover"].append(val_metrics["ssim_cover"])
        history["ssim_secret"].append(val_metrics["ssim_secret"])

        elapsed = time.time() - t0
        print(f"Epoch [{epoch+1}/{cfg.train.epochs}]  ({elapsed:.1f}s)  "
              f"LR={scheduler.get_last_lr()[0]:.1e}")
        print(f"  Train Loss : {train_losses['total']:.4f}  "
              f"(cov_mse={train_losses['cover_mse']:.4f}  "
              f"sec_mse={train_losses['secret_mse']:.4f}  "
              f"sec_ssim={train_losses['secret_ssim']:.4f}  "
              f"percep_cov={train_losses['percep_cover']:.4f}  "
              f"percep_sec={train_losses['percep_secret']:.4f})")
        print(f"  Val Loss   : {val_losses['total']:.4f}")
        print(f"  " + metrics_calc.format(val_metrics))

        # Save best model
        if val_metrics["psnr_secret"] > best_psnr_secret:
            best_psnr_secret = val_metrics["psnr_secret"]
            save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                            os.path.join(cfg.train.checkpoint_dir, "best_model.pth"))

        # Periodic checkpoint
        if (epoch + 1) % cfg.train.save_every == 0:
            save_checkpoint(model, optimizer, scheduler, epoch, {},
                            os.path.join(cfg.train.checkpoint_dir, f"epoch_{epoch+1:03d}.pth"))

        # Visualization
        if (epoch + 1) % cfg.train.vis_every == 0 and vis_batch:
            show_comparison(*vis_batch, n=cfg.train.n_vis_samples,
                            title=f"Epoch {epoch+1} — Cover | Stego | Secret | Revealed")
            save_vis_grid(*vis_batch,
                          path=os.path.join(cfg.train.vis_dir, f"epoch_{epoch+1:03d}.png"),
                          n=cfg.train.n_vis_samples)

    print(f"✓ Training complete. Best PSNR (secret): {best_psnr_secret:.2f} dB")
    return model, history


# ── Run training ─────────────────────────────────────────────
model, history = run_training(cfg)

  Deep Steganography Training
  Device      : cuda
  Epochs      : 10
  Batch size  : 8
  Train imgs  : 5000
  Val imgs    : 200
Parameters: 15,648,942
[StegoDataset] train: 5000 images from '/content/drive/MyDrive/image-in-image/data/imagenet/train'
[StegoDataset] val: 200 images from '/content/drive/MyDrive/image-in-image/data/imagenet/val'


Epoch 1 [Train]:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
  # ── Plot training history ─────────────────────────────────────
def plot_history(history: dict):
    """Plot loss curves and PSNR/SSIM metrics after training."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss
    axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o", markersize=3)
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker="o", markersize=3)
    axes[0].set_title("Loss Curves"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # PSNR
    axes[1].plot(epochs, history["psnr_cover"],  label="PSNR Cover (C↔C')",    marker="o", markersize=3)
    axes[1].plot(epochs, history["psnr_secret"], label="PSNR Secret (S↔S')",   marker="o", markersize=3)
    axes[1].axhline(y=33, color="green",  linestyle="--", alpha=0.5, label="Target cover ≥33dB")
    axes[1].axhline(y=28, color="orange", linestyle="--", alpha=0.5, label="Target secret ≥28dB")
    axes[1].set_title("PSNR (higher = better)"); axes[1].set_xlabel("Epoch")
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    # SSIM
    axes[2].plot(epochs, history["ssim_cover"],  label="SSIM Cover (C↔C')",   marker="o", markersize=3)
    axes[2].plot(epochs, history["ssim_secret"], label="SSIM Secret (S↔S')",  marker="o", markersize=3)
    axes[2].axhline(y=0.95, color="green",  linestyle="--", alpha=0.5, label="Target cover ≥0.95")
    axes[2].axhline(y=0.90, color="orange", linestyle="--", alpha=0.5, label="Target secret ≥0.90")
    axes[2].set_title("SSIM (higher = better)"); axes[2].set_xlabel("Epoch")
    axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

    plt.suptitle("Training History", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Call after training
plot_history(history)

---
## Section 8 — Inference & Demo

After training, use these cells to:
1. Load a saved checkpoint
2. Hide a secret image inside a cover image
3. Reveal the hidden secret
4. Visualize and evaluate all four images with metrics


In [ ]:
# ── Inference helpers ─────────────────────────────────────────

def load_image(path: str, size: int = 256) -> torch.Tensor:
    """Load an image file as (1, 3, H, W) tensor in [0, 1]."""
    transform = T.Compose([T.Resize((size, size)), T.ToTensor()])
    return transform(Image.open(path).convert("RGB")).unsqueeze(0)


def save_image_file(tensor: torch.Tensor, path: str):
    """Save a tensor as an image file."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    vutils.save_image(tensor.squeeze(0).clamp(0, 1), path)
    print(f"  Saved → {path}")


def load_model_from_checkpoint(checkpoint_path: str, cfg: Config) -> StegaNet:
    """Load a trained StegaNet from a .pth checkpoint file."""
    state = torch.load(checkpoint_path, map_location=device)
    model = StegaNet(
        prep_out_ch=cfg.model.prep_out_ch,
        unet_base_ch=cfg.model.unet_base_ch
    ).to(device)
    model.load_state_dict(state["model"])
    model.eval()
    print(f"Model loaded from '{checkpoint_path}'")
    if "metrics" in state and state["metrics"]:
        print(f"  Checkpoint metrics: {state['metrics']}")
    return model


print("Inference helpers defined.")

In [ ]:
# ── Demo: Full Pipeline ───────────────────────────────────────

@torch.no_grad()
def run_demo(cover_path: str, secret_path: str,
             checkpoint_path: str, output_dir: str = "outputs/demo"):
    """
    Full pipeline demo:
      1. Load cover + secret images
      2. Hide secret inside cover → stego
      3. Reveal hidden secret from stego → revealed
      4. Compute metrics + display results
      5. Save all outputs
    """
    # Load model
    model = load_model_from_checkpoint(checkpoint_path, cfg)

    # Load images
    cover  = load_image(cover_path).to(device)
    secret = load_image(secret_path).to(device)
    print(f"Loaded cover : {cover_path}")
    print(f"Loaded secret: {secret_path}")

    # Forward pass
    stego, revealed = model(cover, secret)

    # Metrics
    calc = MetricsCalculator(device=str(device))
    metrics = calc.compute(cover, stego, secret, revealed)
    print("" + "─" * 55)
    print(calc.format(metrics))

    # Display in notebook
    show_comparison(cover.cpu(), stego.cpu(), secret.cpu(), revealed.cpu(),
                    n=1, title="Demo: Cover | Stego | Secret | Revealed")

    # Save outputs
    os.makedirs(output_dir, exist_ok=True)
    save_image_file(cover,    f"{output_dir}/1_cover.png")
    save_image_file(stego,    f"{output_dir}/2_stego.png")
    save_image_file(secret,   f"{output_dir}/3_secret.png")
    save_image_file(revealed, f"{output_dir}/4_revealed.png")

    # Save comparison grid
    grid = torch.cat([cover, stego, secret, revealed], dim=0)
    save_image_file(
        vutils.make_grid(grid, nrow=4, padding=4, normalize=False),
        f"{output_dir}/comparison_grid.png"
    )
    print(f"All outputs saved to '{output_dir}/'")
    return metrics


# ── Run the demo ──────────────────────────────────────────────
metrics = run_demo(
    cover_path      = "/content/drive/MyDrive/image-in-image/data/imagenet/test/000017.jpg",
    secret_path     = "/content/drive/MyDrive/image-in-image/data/imagenet/test/000029.jpg",
    checkpoint_path = "/content/drive/MyDrive/image-in-image/checkpoints/best_model.pth",
    output_dir      = "outputs/demo",
)